***Static Ingestion***

In [0]:
df = spark.read.format("csv")\
              .option("header", "true")\
              .option("inferSchema", "true")\
              .load("/Volumes/pysparkdbt/source/source_data/sch_loan/")

sch_loan_schema=df.schema

***Dynamic Ingestion SPARK STREAMING***

In [0]:

files = ["sch_loan", "Loan_data", "repayment_data"]

for file in files:

    # Read batch data to infer schema
    batch_df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .load(f"/Volumes/pysparkdbt/source/source_data/{file}/")

    batch_schema = batch_df.schema

    # Read data as a stream
    df = spark.readStream.format("csv") \
        .option("header", "true") \
        .schema(batch_schema) \
        .load(f"/Volumes/pysparkdbt/source/source_data/{file}/")

    # Write streaming data to Bronze Delta table
    df.writeStream.format("delta") \
        .outputMode("append") \
        .option(
            "checkpointLocation",
            f"/Volumes/pysparkdbt/bronze/checkpoint/{file}"
        ) \
        .trigger(once=True) \
        .toTable(f"pysparkdbt.bronze.{file}")
